# Low two-sided bulkKS による保守的 LRA と LoRA 回復実験

## 目的

両端の外れ値を除いた MP bulk との KS 距離が小さい行列だけを候補にし、同じ行列集合へ **DEあり / DEなし** の低ランク近似を適用する。その直後と LoRA fine-tuning 後の WikiText-2 PPL を比較する。

対象範囲は次の3通り。

1. `mlp`: `gate_proj`, `up_proj`, `down_proj`
2. `mlp_v`: MLP + `v_proj`
3. `all`: MLP + attention の `q/k/v/o_proj`

行列選択には、`04_two_sided_bulk_ks.ipynb` の主解析である `method=matched_upper` の `two_sided_bulkKS` を使う。DE前はBEMA推定分散を使う `condition=preDE`、DE後は分散1を使う `condition=DE_1` とする。

### 比較を解釈するための固定事項

- `shared_postDE`: DE後（分散1）のbulkKSで同じ行列を選び、変換だけの差を見る。
- `condition_specific`: DEなしはpreDE＋BEMA、DEありはpostDE＋分散1で別々に選び、選択を含むパイプライン全体の差を見る。
- ランクだけを変える。DEありは保存済み `s_hat_saved_postDE`、DEなしは `s_hat_saved_preDE` を使う。
- LoRA条件は0901と合わせる（WikiText-2、1 epoch、lr=2e-4、r=8、alpha=16、dropout=0.05）。
- MP閾値から得たランクをそのまま使う従来条件に加え、`max(MP推定ランク, 最小保持ランク)` とする保守的条件を比較する。行列選択自体は変えない。
- PPLを測るのは未圧縮基準、LRA直後、LoRA後。LRA中の各行列ごとのPPLは測らない。
- 表示する削減率は低ランク因子として保存した場合の理論値であり、このノートブック内のモデルは比較のため密行列へ再構成している。

> このノートブックは実行前の状態で保存している。まず設定・計画表を確認し、`RUN_EXPERIMENTS=True` にして本番セルを実行する。


## 1. Colab / VS Code セットアップ

推奨GPUは **L4**。この実験は反復回数が多いため、通常はA100を使う必要はない。VS CodeからColabカーネルへ接続して実行する場合も、下のDriveマウントと認証セルはColab側で実行される。


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

INSTALL_DEPENDENCIES = True

if INSTALL_DEPENDENCIES:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        "transformers>=4.51,<5", "datasets>=3,<5", "peft>=0.15,<1",
        "accelerate>=1,<2", "wandb>=0.19,<1", "sentencepiece", "scipy", "pandas",
    ])

try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError as exc:
    raise RuntimeError("Google Colabカーネルで実行してください。") from exc

REPO_DIR = Path("/content/RMT_utils")
if not REPO_DIR.exists():
    subprocess.check_call([
        "git", "clone", "--depth", "1",
        "https://github.com/ryy1210/RMT_utils.git", str(REPO_DIR),
    ])

if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

print("RMT_utils:", REPO_DIR)


### APIキー

VS Code接続のColabではColab Secretsが見えない場合があるため、このセルで文字列として明示的に渡す。**値を入れた状態のノートブックをGitHubへcommitしないこと。** W&Bを使わない場合は `PUBLISH_WANDB=False` のままで、W&Bキーは空欄でよい。


In [ ]:
# 実行直前に自分の値へ置換する。キーは出力しない。
HF_TOKEN = "PASTE_HF_TOKEN_HERE"
WANDB_API_KEY = "PASTE_WANDB_API_KEY_HERE"

if HF_TOKEN and not HF_TOKEN.startswith("PASTE_"):
    os.environ["HF_TOKEN"] = HF_TOKEN
if WANDB_API_KEY and not WANDB_API_KEY.startswith("PASTE_"):
    os.environ["WANDB_API_KEY"] = WANDB_API_KEY

if "HF_TOKEN" not in os.environ:
    raise RuntimeError("HF_TOKENをこのセルへ明示的に入力してください。")
print("HF_TOKEN: configured")
print("WANDB_API_KEY:", "configured" if "WANDB_API_KEY" in os.environ else "not configured")


## 2. 実験設定

`KS_THRESHOLDS`、`MAX_LRA_LAYERS`、`MIN_RETAINED_RANKS` は直積で展開される。まずは既定値で確認し、必要なら増やす。

- 閾値だけで選ぶ: `MAX_LRA_LAYERS = [None]`
- 個数だけで選ぶ: `KS_THRESHOLDS = [None]`
- 両方で保守的に絞る: 現在の既定値
- 従来どおりのMP推定ランク: `MIN_RETAINED_RANKS` の `None`
- SQUEEZEを参考にしたランク下限: `MIN_RETAINED_RANKS` の整数。使用ランクを `max(MP推定ランク, 下限)` とする

`MODEL_KEYS` は一度に両モデルを順番に処理できるが、片方だけ再開する場合はリストを1要素にする。


In [ ]:
from itertools import product

EXPERIMENT_TAG = "low-two-sided-bulkks-lra-lora-v2-rank-floor"
MODEL_KEYS = ["llama", "gemma"]
SCOPES = ["mlp", "mlp_v", "all"]

# 選択指標: 04 notebookのprimary設定。
SELECTION_METHOD = "matched_upper"
SELECTION_METRIC = "two_sided_bulkKS"
SELECTION_CONDITIONS = {
    "preDE_bema": "preDE",
    "postDE_variance1": "DE_1",
}
TARGET_SELECTION_MODES = ["shared_postDE", "condition_specific"]

# 例: [0.03, 0.05] や [10, 20, 50] にすればsweepできる。
KS_THRESHOLDS = [0.05]
MAX_LRA_LAYERS = [20]
# Noneは従来計画をそのまま保持する。64は追加する保守的なランク下限。
MIN_RETAINED_RANKS = [None, 64]
DE_CONDITIONS = [True, False]

PPL_DATASET_NAME = "wikitext2"
PPL_SEQ_LEN = 1024
PPL_BATCH_SIZE = 2
TRAIN_SEQ_LEN = 1024
NUM_TRAIN_EPOCHS = 1
LEARNING_RATE = 2e-4
PER_DEVICE_TRAIN_BATCH_SIZE = 1
GRADIENT_ACCUMULATION_STEPS = 8
LORA_R = 8
LORA_ALPHA = 16
LORA_DROPOUT = 0.05
SEED = 42

FAST_SVD = False  # 0901と同じ完全SVD。近似SVDを試す場合だけTrue。
ZERO_RANK_POLICY = "error"  # "error" / "allow"
SAVE_ADAPTER = False
PUBLISH_WANDB = False
WANDB_ENTITY = "ryoya-zushi1210-tokyo-university-of-science"
WANDB_PROJECT = "Low-KS-LRA-Finetune"
SKIP_COMPLETED = True
REUSE_IDENTICAL_PIPELINES = True
RUN_EXPERIMENTS = False  # 計画表を確認したあとTrueへ変更
AUTO_DISCONNECT = True

MODEL_CONFIGS = {
    "llama": {
        "model_id": "meta-llama/Llama-3.2-3B",
        "dtype": "float16",
        "bulk_subdir": "llama",
    },
    "gemma": {
        "model_id": "google/gemma-3-4b-pt",
        "dtype": "bfloat16",
        "bulk_subdir": "gemma",
    },
}

SCOPE_MODULES = {
    "mlp": ["gate_proj", "up_proj", "down_proj"],
    "mlp_v": ["gate_proj", "up_proj", "down_proj", "v_proj"],
    "all": ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
}

DRIVE_DATA = Path("/content/drive/MyDrive/TUS/hashiguchi/data")
TWO_SIDED_PATH = DRIVE_DATA / "bulk_ks_two_sided" / "two-sided-bulk-ks-v1" / "two_sided_bulk_ks.csv"
BULK_ROOT = DRIVE_DATA / "bulk_ks"
SOURCE_TAG = "bulk-ks-20260925-v1-full"
OUTPUT_ROOT = DRIVE_DATA / "low_ks_lra_finetune" / EXPERIMENT_TAG
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

print("output:", OUTPUT_ROOT)
print("planned runs:", len(MODEL_KEYS) * len(TARGET_SELECTION_MODES) * len(SCOPES) * len(KS_THRESHOLDS) * len(MAX_LRA_LAYERS) * len(MIN_RETAINED_RANKS) * len(DE_CONDITIONS))


## 3. 両端bulkKS・保存済みランクの読み込み

`two_sided_bulk_ks.csv` から選択指標を、元の `bulk_ks.csv` から0901と同じ保存済みランクを読み込む。名前の重複、行列集合、ランク再現性をここで検査する。


In [ ]:
import hashlib
import json
import math
import random
import re
import shutil
import time
import warnings

import numpy as np
import pandas as pd


def file_sha256(path):
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def atomic_write_csv(frame, path):
    path = Path(path)
    temp = path.with_suffix(".tmp.csv")
    frame.to_csv(temp, index=False)
    temp.replace(path)


def load_model_metrics(model_key, selection_condition):
    if model_key not in MODEL_CONFIGS:
        raise KeyError(model_key)
    two_sided_all = pd.read_csv(TWO_SIDED_PATH)
    selected_metric = two_sided_all.loc[
        (two_sided_all["model"] == model_key)
        & (two_sided_all["method"] == SELECTION_METHOD)
        & (two_sided_all["condition"] == selection_condition)
    ].copy()
    if selected_metric["name"].duplicated().any():
        raise ValueError(f"{model_key}: two-sided bulkKSに重複名があります")

    rank_path = BULK_ROOT / MODEL_CONFIGS[model_key]["bulk_subdir"] / SOURCE_TAG / "bulk_ks.csv"
    ranks = pd.read_csv(rank_path)
    if ranks["name"].duplicated().any():
        raise ValueError(f"{model_key}: rank表に重複名があります")
    if not ranks[["rank_matches_saved_preDE", "rank_matches_saved_postDE"]].all().all():
        raise ValueError(f"{model_key}: 保存済みランクと再計算ランクが一致しません")
    if set(selected_metric["name"]) != set(ranks["name"]):
        raise ValueError(f"{model_key}: two-sided bulkKS表とrank表の行列集合が一致しません")

    rank_columns = [
        "name", "s_hat_saved_preDE", "s_hat_saved_postDE",
        "p", "n", "module_type", "block_index",
    ]
    merged = selected_metric.merge(
        ranks[rank_columns], on="name", how="inner", validate="one_to_one",
        suffixes=("", "_rank"),
    )
    if not (merged["module_type"] == merged["module_type_rank"]).all():
        raise ValueError(f"{model_key}: module_typeが一致しません")
    if not np.isfinite(merged[SELECTION_METRIC]).all():
        raise ValueError(f"{model_key}: 選択指標にNaN/Infがあります")
    for column in ["s_hat_saved_preDE", "s_hat_saved_postDE"]:
        if (merged[column] < 0).any() or (merged[column] > merged[["p_rank", "n_rank"]].min(axis=1)).any():
            raise ValueError(f"{model_key}: {column} が行列shapeの範囲外です")
    return merged, {
        "two_sided_csv_sha256": file_sha256(TWO_SIDED_PATH),
        "rank_csv_sha256": file_sha256(rank_path),
        "rank_path": str(rank_path),
    }


METRICS = {}
SOURCE_IDENTITIES = {}
for model_key in MODEL_KEYS:
    METRICS[model_key] = {}
    identities = {}
    for condition_label, condition in SELECTION_CONDITIONS.items():
        METRICS[model_key][condition], identity = load_model_metrics(model_key, condition)
        identities[condition_label] = identity
        print(model_key, condition_label, "matrices:", len(METRICS[model_key][condition]))
    SOURCE_IDENTITIES[model_key] = identities


## 4. 実験計画を確定する

候補を両端bulkKSの昇順に並べ、KS閾値と最大行列数を順に適用する。`shared_postDE` では両条件がDE後・分散1のリストを共有する。`condition_specific` ではDEなしがpreDE・BEMA、DEありがpostDE・分散1からそれぞれ選ぶ。

同じモデル内で、異なるscopeや選択方式が結果として同一リストを選ぶ場合がある。`REUSE_IDENTICAL_PIPELINES=True` なら先行runの結果を再利用し、同じモデルを再学習しない。


In [ ]:
def threshold_label(value):
    return "none" if value is None else str(value).replace(".", "p")


def count_label(value):
    return "all" if value is None else str(int(value))


def rank_floor_label(value):
    return "none" if value is None else str(int(value))


def select_layers(metrics, scope, ks_threshold, max_lra_layers):
    if scope not in SCOPE_MODULES:
        raise KeyError(scope)
    selected = metrics.loc[metrics["module_type"].isin(SCOPE_MODULES[scope])].copy()
    if ks_threshold is not None:
        selected = selected.loc[selected[SELECTION_METRIC] <= float(ks_threshold)]
    selected = selected.sort_values([SELECTION_METRIC, "name"], kind="stable")
    if max_lra_layers is not None:
        selected = selected.head(int(max_lra_layers))
    if selected.empty:
        raise ValueError(f"候補が0件です: scope={scope}, ks={ks_threshold}, max={max_lra_layers}")
    return selected.reset_index(drop=True)


def selection_condition_for(selection_mode, use_de):
    if selection_mode == "shared_postDE":
        return SELECTION_CONDITIONS["postDE_variance1"]
    if selection_mode == "condition_specific":
        return (
            SELECTION_CONDITIONS["postDE_variance1"]
            if use_de else SELECTION_CONDITIONS["preDE_bema"]
        )
    raise KeyError(selection_mode)


def build_plan():
    rows = []
    selection_owner = {}
    for model_key, selection_mode, scope, ks_threshold, max_layers, min_rank, use_de in product(
        MODEL_KEYS, TARGET_SELECTION_MODES, SCOPES, KS_THRESHOLDS, MAX_LRA_LAYERS,
        MIN_RETAINED_RANKS, DE_CONDITIONS,
    ):
        selection_condition = selection_condition_for(selection_mode, use_de)
        selected = select_layers(METRICS[model_key][selection_condition], scope, ks_threshold, max_layers)
        signature = tuple(selected["name"])
        rank_col = "s_hat_saved_postDE" if use_de else "s_hat_saved_preDE"
        estimated_ranks = selected[rank_col].astype(int).to_numpy()
        matrix_ranks = selected[["p_rank", "n_rank"]].min(axis=1).astype(int).to_numpy()
        floor_values = np.zeros(len(selected), dtype=int) if min_rank is None else np.minimum(int(min_rank), matrix_ranks)
        effective_ranks = np.maximum(estimated_ranks, floor_values)
        pipeline_items = [f"{name}:{rank}" for name, rank in zip(signature, effective_ranks)]
        pipeline_signature = hashlib.sha256("\n".join(pipeline_items).encode()).hexdigest()
        owner_key = (model_key, selection_mode, ks_threshold, max_layers, use_de, pipeline_signature)
        duplicate_of = selection_owner.get(owner_key)
        if duplicate_of is None:
            selection_owner[owner_key] = scope
        run_id = (
            f"{model_key}__{selection_mode}__{scope}__ks-{threshold_label(ks_threshold)}"
            f"__n-{count_label(max_layers)}__floor-{rank_floor_label(min_rank)}"
            f"__{'DE' if use_de else 'OffDE'}"
        )
        rows.append({
            "run_id": run_id,
            "model": model_key,
            "selection_mode": selection_mode,
            "selection_condition": selection_condition,
            "scope": scope,
            "ks_threshold": ks_threshold,
            "max_lra_layers": max_layers,
            "min_retained_rank": min_rank,
            "rank_floor_label": rank_floor_label(min_rank),
            "use_de": use_de,
            "rank_column": rank_col,
            "selected_count": len(selected),
            "selected_min_ks": selected[SELECTION_METRIC].min(),
            "selected_max_ks": selected[SELECTION_METRIC].max(),
            "rank_floor_applied_count": int((effective_ranks > estimated_ranks).sum()),
            "zero_rank_count": int((effective_ranks == 0).sum()),
            "duplicate_selection_of_scope": duplicate_of,
            "selection_signature": hashlib.sha256("\n".join(signature).encode()).hexdigest(),
            "pipeline_signature": pipeline_signature,
        })
    return pd.DataFrame(rows)


PLAN = build_plan()
display_columns = [
    "run_id", "selection_condition", "selected_count", "selected_min_ks", "selected_max_ks",
    "min_retained_rank", "rank_floor_applied_count", "zero_rank_count", "duplicate_selection_of_scope",
]
display(PLAN[display_columns].style.format({
    "selected_min_ks": "{:.4f}", "selected_max_ks": "{:.4f}",
}, na_rep="-"))

if PLAN["zero_rank_count"].gt(0).any() and ZERO_RANK_POLICY == "error":
    warnings.warn("rank=0を含む計画があります。本番セルは該当runで停止します。")
if PLAN["duplicate_selection_of_scope"].notna().any():
    print("注意: 同じ選択リストのscopeがあります。既定では先行runの結果を再利用します。")

for model_key in MODEL_KEYS:
    for condition_label, condition in SELECTION_CONDITIONS.items():
        example = select_layers(METRICS[model_key][condition], SCOPES[0], KS_THRESHOLDS[0], MAX_LRA_LAYERS[0])
        print("\n", model_key, "/", condition_label, "/", SCOPES[0], "module counts")
        display(example["module_type"].value_counts().rename("selected").to_frame())
        display(example[["name", "module_type", SELECTION_METRIC, "s_hat_saved_preDE", "s_hat_saved_postDE"]].head(20))


## 5. LRA・PPL・LoRA関数

各runはfresh modelから始める。モデル名は保存表の名前と完全一致しないことがあるため、唯一のsuffix一致で解決し、曖昧な場合は停止する。


In [ ]:
import gc

import torch
import torch.nn as nn
from datasets import load_dataset
from peft import LoraConfig, TaskType, get_peft_model
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    DataCollatorForLanguageModeling,
    Trainer,
    TrainingArguments,
    set_seed,
)

from funcs1 import apply_lra_1, get_ppl


def reset_seed(seed=SEED):
    set_seed(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def torch_dtype_for(model_key):
    return getattr(torch, MODEL_CONFIGS[model_key]["dtype"])


def load_tokenizer(model_key):
    tokenizer = AutoTokenizer.from_pretrained(
        MODEL_CONFIGS[model_key]["model_id"], token=os.environ["HF_TOKEN"],
    )
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token = tokenizer.eos_token
    return tokenizer


def load_fresh_model(model_key):
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_CONFIGS[model_key]["model_id"],
        token=os.environ["HF_TOKEN"],
        torch_dtype=torch_dtype_for(model_key),
        low_cpu_mem_usage=True,
        trust_remote_code=True,
        device_map="cpu",
    )
    model.config.use_cache = False
    return model


def resolve_linear_module(model, metric_name):
    linear_modules = {name: module for name, module in model.named_modules() if isinstance(module, nn.Linear)}
    if metric_name in linear_modules:
        return metric_name, linear_modules[metric_name]
    matches = [(name, module) for name, module in linear_modules.items() if name.endswith("." + metric_name)]
    if len(matches) != 1:
        raise KeyError(f"{metric_name}: Linearのsuffix一致が{len(matches)}件です")
    return matches[0]


@torch.no_grad()
def apply_selected_lra(model, selected, use_de, min_retained_rank=None):
    rank_col = "s_hat_saved_postDE" if use_de else "s_hat_saved_preDE"
    if min_retained_rank is not None and (int(min_retained_rank) != min_retained_rank or min_retained_rank < 0):
        raise ValueError("min_retained_rank must be a nonnegative integer or None")

    records = []
    total_model_params = sum(parameter.numel() for parameter in model.parameters())
    total_saved = 0
    for index, row in selected.iterrows():
        resolved_name, module = resolve_linear_module(model, row["name"])
        estimated_rank = int(row[rank_col])
        m, n = module.weight.shape
        rank_floor = 0 if min_retained_rank is None else min(int(min_retained_rank), min(m, n))
        rank = max(estimated_rank, rank_floor)
        if ZERO_RANK_POLICY == "error" and rank == 0:
            raise ValueError(f"rank=0が含まれます: {row['name']}")
        if not 0 <= rank <= min(m, n):
            raise ValueError(f"{row['name']}: invalid rank {rank} for {(m, n)}")
        before = module.weight.detach()
        approximation = apply_lra_1(before, rank, DE=use_de, fast_SVD=FAST_SVD)
        if not torch.isfinite(approximation).all():
            raise FloatingPointError(f"{row['name']}: LRA result has NaN/Inf")
        module.weight.copy_(approximation)

        original_params = m * n
        factorized_params = rank * (m + n)
        saved = max(original_params - factorized_params, 0)
        total_saved += saved
        records.append({
            "order": index + 1,
            "metric_name": row["name"],
            "resolved_name": resolved_name,
            "module_type": row["module_type"],
            SELECTION_METRIC: float(row[SELECTION_METRIC]),
            "use_de": bool(use_de),
            "rank_column": rank_col,
            "estimated_rank": estimated_rank,
            "min_retained_rank": min_retained_rank,
            "rank_floor_applied": bool(rank > estimated_rank),
            "rank": rank,
            "rows": int(m),
            "columns": int(n),
            "original_params": int(original_params),
            "factorized_params": int(factorized_params),
            "params_saved": int(saved),
        })
        floor_note = f" (estimated={estimated_rank}, floor applied)" if rank > estimated_rank else ""
        print(f"[{index + 1}/{len(selected)}] {'DE' if use_de else 'OffDE'} LRA: {row['name']} rank={rank}{floor_note}")
    history = pd.DataFrame(records)
    theoretical_reduction = 100.0 * total_saved / total_model_params
    return history, theoretical_reduction


def evaluate_ppl(model, tokenizer):
    model.eval()
    return float(get_ppl(
        model, tokenizer, dataset_name=PPL_DATASET_NAME,
        seq_len=PPL_SEQ_LEN, batch_size=PPL_BATCH_SIZE,
    ))


def prepare_train_dataset(tokenizer):
    raw = load_dataset("Salesforce/wikitext", "wikitext-2-raw-v1", split="train")
    raw = raw.filter(lambda example: example["text"] is not None and example["text"].strip() != "")

    def tokenize(examples):
        return tokenizer(examples["text"], add_special_tokens=False)

    tokenized = raw.map(
        tokenize, batched=True, remove_columns=raw.column_names,
        num_proc=2, desc="Tokenizing WikiText-2 train",
    )

    def group_texts(examples):
        concatenated = {key: sum(examples[key], []) for key in examples}
        total = (len(concatenated["input_ids"]) // TRAIN_SEQ_LEN) * TRAIN_SEQ_LEN
        return {
            key: [values[i:i + TRAIN_SEQ_LEN] for i in range(0, total, TRAIN_SEQ_LEN)]
            for key, values in concatenated.items()
        }

    return tokenized.map(
        group_texts, batched=True, num_proc=2,
        desc=f"Grouping into {TRAIN_SEQ_LEN}-token blocks",
    )


LORA_TARGET_MODULES = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]


def attach_lora(model):
    config = LoraConfig(
        task_type=TaskType.CAUSAL_LM,
        inference_mode=False,
        r=LORA_R,
        lora_alpha=LORA_ALPHA,
        lora_dropout=LORA_DROPOUT,
        bias="none",
        target_modules=LORA_TARGET_MODULES,
    )
    model = get_peft_model(model, config)
    model.gradient_checkpointing_enable()
    model.enable_input_require_grads()
    model.print_trainable_parameters()
    return model


def train_lora(model, tokenizer, train_dataset, experiment_dir, model_key):
    use_bf16 = torch_dtype_for(model_key) == torch.bfloat16
    args = TrainingArguments(
        output_dir=str(experiment_dir / "trainer"),
        num_train_epochs=NUM_TRAIN_EPOCHS,
        learning_rate=LEARNING_RATE,
        per_device_train_batch_size=PER_DEVICE_TRAIN_BATCH_SIZE,
        gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
        fp16=not use_bf16,
        bf16=use_bf16,
        gradient_checkpointing=True,
        logging_strategy="steps",
        logging_steps=10,
        save_strategy="no",
        report_to="none",
        optim="adamw_torch",
        weight_decay=0.0,
        warmup_ratio=0.03,
        lr_scheduler_type="cosine",
        max_grad_norm=1.0,
        remove_unused_columns=False,
        dataloader_num_workers=2,
        seed=SEED,
        data_seed=SEED,
    )
    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=train_dataset,
        data_collator=DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False),
    )
    result = trainer.train()
    if SAVE_ADAPTER:
        adapter_dir = experiment_dir / "final_adapter"
        model.save_pretrained(adapter_dir)
        tokenizer.save_pretrained(adapter_dir)
    return trainer, dict(result.metrics)


def cleanup(*objects):
    for obj in objects:
        del obj
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()


## 6. 本番実行

このセルは `RUN_EXPERIMENTS=True` のときだけ動く。各run完了時にDriveへ `selected_layers.csv`, `summary.json`, `train_metrics.json` を保存し、全体の `results.csv` も更新する。途中停止後の再実行では成功済みrunを省略する。

同じモデルの未圧縮PPLは最初に一度だけ測る。LoRA学習は各runで同じseed・同じデータを使う。


In [ ]:
import wandb


RESULTS_PATH = OUTPUT_ROOT / "results.csv"
MANIFEST_PATH = OUTPUT_ROOT / "experiment_manifest.json"


def serializable_config():
    return {
        "experiment_tag": EXPERIMENT_TAG,
        "models": MODEL_KEYS,
        "scopes": SCOPES,
        "selection_method": SELECTION_METHOD,
        "selection_conditions": SELECTION_CONDITIONS,
        "target_selection_modes": TARGET_SELECTION_MODES,
        "selection_metric": SELECTION_METRIC,
        "ks_thresholds": KS_THRESHOLDS,
        "max_lra_layers": MAX_LRA_LAYERS,
        "min_retained_ranks": MIN_RETAINED_RANKS,
        "de_conditions": DE_CONDITIONS,
        "fast_svd": FAST_SVD,
        "zero_rank_policy": ZERO_RANK_POLICY,
        "ppl": {"dataset": PPL_DATASET_NAME, "seq_len": PPL_SEQ_LEN, "batch_size": PPL_BATCH_SIZE},
        "lora": {
            "epochs": NUM_TRAIN_EPOCHS, "learning_rate": LEARNING_RATE,
            "rank": LORA_R, "alpha": LORA_ALPHA, "dropout": LORA_DROPOUT,
            "train_seq_len": TRAIN_SEQ_LEN,
            "per_device_batch": PER_DEVICE_TRAIN_BATCH_SIZE,
            "gradient_accumulation": GRADIENT_ACCUMULATION_STEPS,
        },
        "source_identities": SOURCE_IDENTITIES,
    }


manifest = serializable_config()
manifest_json = json.dumps(manifest, ensure_ascii=False, sort_keys=True)
manifest["config_sha256"] = hashlib.sha256(manifest_json.encode()).hexdigest()
if MANIFEST_PATH.exists():
    existing_manifest = json.loads(MANIFEST_PATH.read_text())
    if existing_manifest["config_sha256"] != manifest["config_sha256"]:
        raise RuntimeError("同じEXPERIMENT_TAGに異なる設定があります。TAGを変更してください。")
else:
    MANIFEST_PATH.write_text(json.dumps(manifest, ensure_ascii=False, indent=2))


def load_results():
    if RESULTS_PATH.exists():
        return pd.read_csv(RESULTS_PATH)
    return pd.DataFrame()


def completed_run_ids():
    existing = load_results()
    if existing.empty or "status" not in existing:
        return set()
    return set(existing.loc[existing["status"] == "ok", "run_id"])


def save_result_row(row):
    existing = load_results()
    updated = pd.concat([existing, pd.DataFrame([row])], ignore_index=True)
    updated = updated.drop_duplicates("run_id", keep="last")
    atomic_write_csv(updated, RESULTS_PATH)


def none_if_nan(value):
    return None if pd.isna(value) else value


def run_one_experiment(plan_row, tokenizer, train_dataset, baseline_ppl):
    run_id = plan_row["run_id"]
    model_key = plan_row["model"]
    experiment_dir = OUTPUT_ROOT / run_id
    experiment_dir.mkdir(parents=True, exist_ok=True)
    selected = select_layers(
        METRICS[model_key][plan_row["selection_condition"]], plan_row["scope"],
        none_if_nan(plan_row["ks_threshold"]), none_if_nan(plan_row["max_lra_layers"]),
    )
    selected.to_csv(experiment_dir / "selected_layers.csv", index=False)

    reset_seed()
    model = load_fresh_model(model_key)
    history, theoretical_reduction = apply_selected_lra(
        model, selected, bool(plan_row["use_de"]), none_if_nan(plan_row["min_retained_rank"]),
    )
    history.to_csv(experiment_dir / "lra_history.csv", index=False)

    model.to("cuda")
    ppl_before = evaluate_ppl(model, tokenizer)
    model = attach_lora(model)
    trainer, train_metrics = train_lora(model, tokenizer, train_dataset, experiment_dir, model_key)
    ppl_after = evaluate_ppl(model, tokenizer)

    degradation_before = ppl_before - baseline_ppl
    degradation_after = ppl_after - baseline_ppl
    recovery_ratio = (
        (degradation_before - degradation_after) / degradation_before
        if degradation_before > 0 else np.nan
    )
    row = {
        "run_id": run_id,
        "status": "ok",
        "model": model_key,
        "model_id": MODEL_CONFIGS[model_key]["model_id"],
        "selection_mode": plan_row["selection_mode"],
        "selection_condition": plan_row["selection_condition"],
        "scope": plan_row["scope"],
        "use_de": bool(plan_row["use_de"]),
        "ks_threshold": none_if_nan(plan_row["ks_threshold"]),
        "max_lra_layers": none_if_nan(plan_row["max_lra_layers"]),
        "min_retained_rank": none_if_nan(plan_row["min_retained_rank"]),
        "rank_floor_label": plan_row["rank_floor_label"],
        "rank_floor_applied_count": int(history["rank_floor_applied"].sum()),
        "selected_count": len(selected),
        "selected_min_ks": float(selected[SELECTION_METRIC].min()),
        "selected_max_ks": float(selected[SELECTION_METRIC].max()),
        "selection_signature": plan_row["selection_signature"],
        "pipeline_signature": plan_row["pipeline_signature"],
        "ppl_baseline": float(baseline_ppl),
        "ppl_before_lora": float(ppl_before),
        "ppl_after_lora": float(ppl_after),
        "ppl_degradation_before_lora": float(degradation_before),
        "ppl_degradation_after_lora": float(degradation_after),
        "absolute_ppl_improvement_by_lora": float(ppl_before - ppl_after),
        "recovery_ratio": float(recovery_ratio),
        "theoretical_reduction_ratio_percent": float(theoretical_reduction),
        "train_loss": train_metrics.get("train_loss"),
        "train_runtime_seconds": train_metrics.get("train_runtime"),
        "completed_at": pd.Timestamp.now(tz="Asia/Tokyo").isoformat(),
    }
    (experiment_dir / "summary.json").write_text(json.dumps(row, ensure_ascii=False, indent=2))
    (experiment_dir / "train_metrics.json").write_text(json.dumps(train_metrics, ensure_ascii=False, indent=2, default=str))

    if PUBLISH_WANDB:
        run = wandb.init(
            entity=WANDB_ENTITY, project=WANDB_PROJECT, name=run_id,
            config={**manifest, **{key: row[key] for key in ["model", "selection_mode", "selection_condition", "scope", "use_de", "ks_threshold", "max_lra_layers", "min_retained_rank"]}},
            reinit=True,
        )
        run.log({key: value for key, value in row.items() if isinstance(value, (int, float, bool))})
        run.log({"selected_layers": wandb.Table(dataframe=selected)})
        run.finish()

    del trainer, model
    cleanup()
    return row


def find_reusable_result(plan_row):
    existing = load_results()
    required = {"status", "model", "use_de", "pipeline_signature", "run_id"}
    if existing.empty or not required <= set(existing.columns):
        return None
    candidates = existing.loc[
        (existing["status"] == "ok")
        & (existing["model"] == plan_row["model"])
        & (existing["use_de"].astype(str) == str(bool(plan_row["use_de"])))
        & (existing["pipeline_signature"] == plan_row["pipeline_signature"])
    ]
    return None if candidates.empty else candidates.iloc[-1].to_dict()


def save_reused_result(plan_row, source_row):
    reused = dict(source_row)
    source_run_id = reused["run_id"]
    reused.update({
        "run_id": plan_row["run_id"],
        "selection_mode": plan_row["selection_mode"],
        "selection_condition": plan_row["selection_condition"],
        "scope": plan_row["scope"],
        "ks_threshold": none_if_nan(plan_row["ks_threshold"]),
        "max_lra_layers": none_if_nan(plan_row["max_lra_layers"]),
        "min_retained_rank": none_if_nan(plan_row["min_retained_rank"]),
        "rank_floor_label": plan_row["rank_floor_label"],
        "rank_floor_applied_count": plan_row["rank_floor_applied_count"],
        "pipeline_signature": plan_row["pipeline_signature"],
        "reused_from_run_id": source_run_id,
        "completed_at": pd.Timestamp.now(tz="Asia/Tokyo").isoformat(),
    })
    experiment_dir = OUTPUT_ROOT / plan_row["run_id"]
    experiment_dir.mkdir(parents=True, exist_ok=True)
    selected = select_layers(
        METRICS[plan_row["model"]][plan_row["selection_condition"]],
        plan_row["scope"], none_if_nan(plan_row["ks_threshold"]),
        none_if_nan(plan_row["max_lra_layers"]),
    )
    selected.to_csv(experiment_dir / "selected_layers.csv", index=False)
    (experiment_dir / "summary.json").write_text(json.dumps(reused, ensure_ascii=False, indent=2, default=str))
    save_result_row(reused)
    return reused


ALL_EXPERIMENTS_COMPLETED = False
if not RUN_EXPERIMENTS:
    print("計画のみ作成しました。本番はRUN_EXPERIMENTS=Trueに変更してこのセルを再実行してください。")
else:
    if not torch.cuda.is_available():
        raise RuntimeError("GPUが必要です。ColabのランタイムをL4 GPUへ変更してください。")
    if PUBLISH_WANDB and "WANDB_API_KEY" not in os.environ:
        raise RuntimeError("PUBLISH_WANDB=TrueですがWANDB_API_KEYがありません。")

    completed = completed_run_ids() if SKIP_COMPLETED else set()
    failures = []
    for model_key in MODEL_KEYS:
        tokenizer = load_tokenizer(model_key)
        train_dataset = prepare_train_dataset(tokenizer)

        baseline_id = f"{model_key}__uncompressed_baseline"
        existing = load_results()
        baseline_rows = existing.loc[(existing.get("run_id") == baseline_id)] if not existing.empty else pd.DataFrame()
        if not baseline_rows.empty and baseline_rows.iloc[-1].get("status") == "ok":
            baseline_ppl = float(baseline_rows.iloc[-1]["ppl_baseline"])
            print(model_key, "saved baseline PPL:", baseline_ppl)
        else:
            reset_seed()
            baseline_model = load_fresh_model(model_key).to("cuda")
            baseline_ppl = evaluate_ppl(baseline_model, tokenizer)
            save_result_row({
                "run_id": baseline_id, "status": "ok", "model": model_key,
                "model_id": MODEL_CONFIGS[model_key]["model_id"], "scope": "uncompressed",
                "ppl_baseline": baseline_ppl,
                "completed_at": pd.Timestamp.now(tz="Asia/Tokyo").isoformat(),
            })
            del baseline_model
            cleanup()

        model_plan = PLAN.loc[PLAN["model"] == model_key]
        for _, plan_row in model_plan.iterrows():
            run_id = plan_row["run_id"]
            if SKIP_COMPLETED and run_id in completed:
                print("skip completed:", run_id)
                continue
            reusable = find_reusable_result(plan_row) if REUSE_IDENTICAL_PIPELINES else None
            if reusable is not None:
                print("reuse identical pipeline:", run_id, "from", reusable["run_id"])
                save_reused_result(plan_row, reusable)
                completed.add(run_id)
                continue
            try:
                print("\n" + "=" * 100)
                print("START", run_id)
                result = run_one_experiment(plan_row, tokenizer, train_dataset, baseline_ppl)
                save_result_row(result)
                completed.add(run_id)
            except Exception as exc:
                failures.append(run_id)
                save_result_row({
                    "run_id": run_id, "status": "failed", "model": model_key,
                    "selection_mode": plan_row["selection_mode"],
                    "selection_condition": plan_row["selection_condition"],
                    "scope": plan_row["scope"], "use_de": bool(plan_row["use_de"]),
                    "error_type": type(exc).__name__, "error": str(exc),
                    "completed_at": pd.Timestamp.now(tz="Asia/Tokyo").isoformat(),
                })
                print("FAILED", run_id, type(exc).__name__, exc)
                cleanup()

        del train_dataset, tokenizer
        cleanup()

    ALL_EXPERIMENTS_COMPLETED = len(failures) == 0
    if failures:
        raise RuntimeError(f"失敗したrunがあります。自動切断しません: {failures}")
    print("All scheduled experiments completed.")


## 7. 結果の集計

DE差は `DE - OffDE` で定義する。負ならDEありのPPLが低い。LRA直後とLoRA後を分けて見る。


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns


if RESULTS_PATH.exists():
    results = pd.read_csv(RESULTS_PATH)
    experiment_results = results.loc[
        (results["status"] == "ok") & results["scope"].isin(SCOPES)
    ].copy()
    if experiment_results.empty:
        print("完了済みの本番結果はまだありません。")
    else:
        key_columns = ["model", "selection_mode", "scope", "ks_threshold", "max_lra_layers", "rank_floor_label"]
        paired = experiment_results.pivot_table(
            index=key_columns, columns="use_de",
            values=["ppl_before_lora", "ppl_after_lora", "theoretical_reduction_ratio_percent"],
            aggfunc="last",
        )
        paired.columns = [f"{metric}_{'DE' if use_de else 'OffDE'}" for metric, use_de in paired.columns]
        paired = paired.reset_index()
        paired["delta_ppl_before_DE_minus_OffDE"] = paired["ppl_before_lora_DE"] - paired["ppl_before_lora_OffDE"]
        paired["delta_ppl_after_DE_minus_OffDE"] = paired["ppl_after_lora_DE"] - paired["ppl_after_lora_OffDE"]
        display(paired.sort_values(["model", "ks_threshold", "max_lra_layers", "scope"]))

        plot_data = paired.melt(
            id_vars=["model", "selection_mode", "scope", "ks_threshold", "max_lra_layers", "rank_floor_label"],
            value_vars=["delta_ppl_before_DE_minus_OffDE", "delta_ppl_after_DE_minus_OffDE"],
            var_name="stage", value_name="PPL_DE_minus_OffDE",
        )
        plot_data["stage"] = plot_data["stage"].map({
            "delta_ppl_before_DE_minus_OffDE": "after LRA / before LoRA",
            "delta_ppl_after_DE_minus_OffDE": "after LoRA",
        })
        plot_data["selection_variant"] = (
            plot_data["selection_mode"] + " / floor=" + plot_data["rank_floor_label"].astype(str)
        )
        sns.set_theme(style="whitegrid", context="notebook")
        chart = sns.catplot(
            data=plot_data, x="scope", y="PPL_DE_minus_OffDE",
            hue="stage", col="model", row="selection_variant", kind="bar", height=3.6, aspect=1.15,
        )
        for axis in chart.axes.flat:
            axis.axhline(0, color="black", linewidth=1)
            axis.set_ylabel("PPL difference: DE - OffDE (negative favors DE)")
        chart.fig.suptitle("Effect of Dyson Equalization for low two-sided bulkKS selections", y=1.04)
        plt.show()
else:
    print("results.csvはまだありません。")


## 8. 正常終了時のみColabを切断

本番セルがすべて成功した場合だけ切断する。失敗が1件でもあればカーネルを残す。


In [ ]:
if RUN_EXPERIMENTS and AUTO_DISCONNECT:
    if not ALL_EXPERIMENTS_COMPLETED:
        raise RuntimeError("未完了または失敗があるため切断しません。")
    from google.colab import runtime
    print("全runが完了しました。Colabカーネルを切断します。")
    runtime.unassign()
else:
    print("自動切断は行いません。")
